In [1]:
import pandas as pd
import os

In [2]:
import importlib

import utils.conexao
importlib.reload(utils.conexao)
from utils.conexao import get_dados_origem, SCHEMA_ORIGEM

# consulta
query = f'SELECT * FROM "{SCHEMA_ORIGEM}"."servico";'
df_servicos = get_dados_origem(query) 

if df_servicos is not None:
    print("Extração concluída:")
    print(df_servicos.head())

ModuleNotFoundError: No module named 'utils'

In [ ]:
# Padronizar e tratar datas - Remover espaços e garantir tipo string
df_servicos['data_servico'] = df_servicos['data_servico'].astype(str).str.strip()
display(df_servicos)

,id_servico,data_servico,descricao_servico,nome_servico
0,1,2025-06-23,Manutenção de sistema,Manutenção
1,2,23/06/2025,Atualização de software,Atualização
2,3,2025/06/25,Implantação de sistema,Implantação
3,4,2025-02-30,Serviço inválido,Serviço Errado
4,5,2024-12-01,Auditoria de segurança,Auditoria
5,6,12-10-2024,Desenvolvimento de app mobile,Desenvolvimento Mobile
6,7,2023/11/15,Treinamento de equipe,Treinamento
7,8,2024-07-31,Consultoria TI,Consultoria


In [ ]:
# Padronizar e tratar datas - Substituir separadores diferentes por hífen
df_servicos['data_servico'] = df_servicos['data_servico'].str.replace(r'[\/]', '-', regex=True)
display(df_servicos)

,id_servico,data_servico,descricao_servico,nome_servico
0,1,2025-06-23,Manutenção de sistema,Manutenção
1,2,23-06-2025,Atualização de software,Atualização
2,3,2025-06-25,Implantação de sistema,Implantação
3,4,2025-02-30,Serviço inválido,Serviço Errado
4,5,2024-12-01,Auditoria de segurança,Auditoria
5,6,12-10-2024,Desenvolvimento de app mobile,Desenvolvimento Mobile
6,7,2023-11-15,Treinamento de equipe,Treinamento
7,8,2024-07-31,Consultoria TI,Consultoria


In [ ]:
# Problemas que foram identificados na data:
# - formato aaaa/mm/dd (ISO)
# - formato dd/mm/aaaa (pt-BR)
# - formato ambíguo 2024-12-01 (ou 2024-01-12) e 12-10-2024 (ou 10-12-2024)
# - data inválida (2025-02-30)

# Transformar as datas em aaaa/mm/dd
# Tentar converter a data com o formato atual
# Se falhar, tenta próximo formato
# Se nenhum formato funcionar, retorna NaT

def converter_com_formatos(data):
    formatos = ['%Y-%m-%d', '%d-%m-%Y']
    for fmt in formatos:
        try:
            return pd.to_datetime(data, format=fmt, errors='raise')
        except (ValueError, TypeError):
            continue
    return pd.NaT

df_servicos['data_servico'] = df_servicos['data_servico'].apply(converter_com_formatos)
display(df_servicos)

,id_servico,data_servico,descricao_servico,nome_servico
0,1,2025-06-23,Manutenção de sistema,Manutenção
1,2,2025-06-23,Atualização de software,Atualização
2,3,2025-06-25,Implantação de sistema,Implantação
3,4,NaT,Serviço inválido,Serviço Errado
4,5,2024-12-01,Auditoria de segurança,Auditoria
5,6,2024-10-12,Desenvolvimento de app mobile,Desenvolvimento Mobile
6,7,2023-11-15,Treinamento de equipe,Treinamento
7,8,2024-07-31,Consultoria TI,Consultoria


In [ ]:
# Verificar o tipo da variável
print(df_servicos['data_servico'].dtype)

datetime64[ns]


In [ ]:
# Renomer o nome do serviço para padronizar
df_servicos.loc[df_servicos['nome_servico'] == 'Desenvolvimento Mobile', 'nome_servico'] = 'Desenvolvimento'
display(df_servicos)

,id_servico,data_servico,descricao_servico,nome_servico
0,1,2025-06-23,Manutenção de sistema,Manutenção
1,2,2025-06-23,Atualização de software,Atualização
2,3,2025-06-25,Implantação de sistema,Implantação
3,4,NaT,Serviço inválido,Serviço Errado
4,5,2024-12-01,Auditoria de segurança,Auditoria
5,6,2024-10-12,Desenvolvimento de app mobile,Desenvolvimento
6,7,2023-11-15,Treinamento de equipe,Treinamento
7,8,2024-07-31,Consultoria TI,Consultoria


In [ ]:
df_servicos.rename(columns=lambda col: "_".join(p.capitalize() for p in col.split("_")), inplace=True)
display(df_servicos)

,Id_Servico,Data_Servico,Descricao_Servico,Nome_Servico
0,1,2025-06-23,Manutenção de sistema,Manutenção
1,2,2025-06-23,Atualização de software,Atualização
2,3,2025-06-25,Implantação de sistema,Implantação
3,4,NaT,Serviço inválido,Serviço Errado
4,5,2024-12-01,Auditoria de segurança,Auditoria
5,6,2024-10-12,Desenvolvimento de app mobile,Desenvolvimento
6,7,2023-11-15,Treinamento de equipe,Treinamento
7,8,2024-07-31,Consultoria TI,Consultoria


In [ ]:
# Uma das datas é inválida (2025-02-30), a descrição do serviço é  'Serviço inválido' e o nome do serviço é 'Serviço Errado'
# Decisão - apagar ou manter ?
#df_servicos = df_servicos.dropna(subset=['data_servico'])

In [ ]:
#df_servicos.to_csv("../ArquivosPostgresql/dados_tratados_postgresql_v2/servicos_postgres_tratado.csv", index=False)

In [ ]:

# variável de ambiente para construir o caminho
caminho_base_arquivos = os.environ.get("CAMINHO_ARQUIVOS_CSV")
caminho_saida = os.path.join(caminho_base_arquivos, "dados_tratados_v2", "servicos_postgres_tratado.csv")

# variável caminho_saida
df_servicos.to_csv(caminho_saida, index=False)